# 02 - Financial Feature Engineering

Build predictive features with explicit financial rationale.


## Feature Families for Financial Forecasting

**Definition**  
Feature engineering transforms raw OHLCV into model-consumable signals.

**Theory**  
Different signal classes capture different market dynamics: trend, momentum, volatility, liquidity, and temporal context.

**Mathematical Intuition**  
Engineered feature matrix `X_t` enriches representational capacity beyond raw price `P_t`.

**Financial Intuition**  
RSI can proxy momentum exhaustion; ATR captures volatility regime; volume z-score flags abnormal participation.

**Business Impact**  
Better features reduce forecast error and improve model stability across regimes.

**Real-World Example**  
Momentum + volatility interactions often explain why naive models fail during turbulent periods.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

df = load_stock_data(PROJECT_ROOT / 'data' / 'apple_stock_data.csv')
feat_cfg = framework.config['features']

features = create_features(
    df,
    lags=feat_cfg['lags'],
    rolling_windows=feat_cfg['rolling_windows'],
    ema_windows=feat_cfg['ema_windows'],
    wma_windows=feat_cfg['wma_windows'],
    momentum_windows=feat_cfg['momentum_windows'],
    include_technical=feat_cfg['include_technical'],
    include_date_features=feat_cfg['include_date_features'],
    include_price_derived=feat_cfg['include_price_derived'],
    dropna=False,
)

print('Raw columns:', df.shape[1])
print('Feature columns:', features.shape[1])
print('Added columns:', features.shape[1] - df.shape[1])
display(features.tail(5))


In [ ]:

selected = [
    'daily_return', 'log_return', 'sma_20', 'ema_20', 'wma_20',
    'rsi_14', 'roc_10', 'momentum_10', 'rolling_std_20', 'atr_14',
    'bb_width_20', 'volume_change', 'volume_ma_20', 'Close_lag_20', 'Volume_lag_20'
]
selected = [c for c in selected if c in features.columns]

corr = features[selected + ['Close']].corr().sort_values('Close', ascending=False)
display(corr[['Close']])

fig, ax = plt.subplots(figsize=(12, 6))
features[selected].dropna().tail(400).plot(ax=ax, alpha=0.8)
ax.set_title('Sample Engineered Feature Dynamics (Last 400 Rows)')
fig.tight_layout()
fig.savefig(OUT / 'plots/02_feature_dynamics.png', dpi=150)
plt.close(fig)



## Feature Interpretation Checklist

- Trend features reduce noise sensitivity.
- Momentum features capture continuation or reversal pressure.
- Volatility features adapt model expectations under turbulence.
- Volume features encode participation and conviction.
- Lag features create autoregressive memory for non-sequence models.
